# Projeto Final: análise das candidaturas da eleição de 2022

**Autores:** Felipe Albanez de Oliveira

## Datasets utilizados
- **Base principal:** consulta_cand_2022_BRASIL.csv
- **Base auxiliar:**  consulta_cand_complementar_2022_BRASIL.csv

## 1. Importando as bibliotecas Pandas e Numpy

In [9]:
# Para facilitar no desenvolvimento, "apelidamos" as bibliotecas no momento da importação
import pandas as pd
import numpy as np
# import matplotlib.pyplot as plt

## 2. Criando o DataFrame

### 2.1. Erro na primeira tentativa de criação do DataFrame

In [ ]:
"""
caminho note pessoal
pasta_arquivo = "/Users/felipealbanez/Github/CAIXAVERSO_DFF/Projeto_Final_TSE_2022/consulta_cand_2022_BRASIL.csv"

caminho Trabalho
pasta_arquivo = "/Users/cxxxxxx/OneDrive - Caixa Economica Federal/Área de Trabalho/CAIXA Verso/Projeto Final/Base de dados/consulta_cand_2022_BRASIL.csv"

df_original = pd.read_csv(pasta_arquivo)
df_original.head()

erro gerado:
UnicodeDecodeError: 'utf-8' codec can't decode byte 0xc7 in position 838: invalid continuation byte

"""

### 2.2. Segunda tentativa - modo "a brasileira"

In [ ]:
""""
sep=';': Os dados usam ponto e vírgula para separar as colunas
decimal=",": números decimais usam vírgula
encoding='latin-1': Evita erros de leitura com acentos e caracteres da língua portuguesa.
"""
# substitua o caminho abaixo pela pasta onde está salvo a base de dados .csv
# não esquecer que o nome do arquivo e a extensão deve estar inclusos

# caminho note pessoal
pasta_arquivo = "/Users/felipealbanez/Github/CAIXAVERSO_DFF/Projeto_Final_TSE_2022/consulta_cand_2022_BRASIL.csv"

# caminho Trabalho
# pasta_arquivo = "/Users/cxxxxxx/OneDrive - Caixa Economica Federal/Área de Trabalho/CAIXA Verso/Projeto Final/Base de dados/consulta_cand_2022_BRASIL.csv"

# cria o dataframe lendo o arquivo .csv
df_original = pd.read_csv(pasta_arquivo, sep=';', decimal=',',encoding='latin-1')

In [ ]:
# vizualisa as 5 primeiras linhas do dataframe
df_original.head()

## 3. Conhecendo o DataFrame

### 3.1. Dimensões e Estrutura do DataFrame

In [ ]:
# shape: informa o número de linhas e colunas
print(f'Shape: {df_original.shape}')
print(f'Quantidade de linhas: {df_original.shape[0]}')
print(f'Quantidade de colunas: {df_original.shape[1]}')

In [ ]:
# info: informa os tipos, faltantes, memória
df_original.info()        

In [ ]:
# describle aula
display(df_original.describe().round(2))
display(df_original.describe(include='object'))

### 3.2 Avaliando as categorizaçãos

In [ ]:
# Mostra os nomes das colunas.
df_original.columns

In [ ]:
# Avaliação de algumas colunas que serão utilizadas
print(df_original['DS_GENERO'].value_counts(dropna=False))
print()
print(df_original['DS_COR_RACA'].value_counts(dropna=False))
print()
print(df_original['DS_GRAU_INSTRUCAO'].value_counts(dropna=False))

## 4. Tratamento de Faltantes (NaN)

### 4.1. Faltantes puros

In [79]:
# cria um df chamado Faltantes
# são criadas tres colunas para o df 
# isna.sum: percorre a coluna e retorna a soma dos faltantes
# isna.mean: percorre a coluna e retorna a média dos faltantes em relação ao número total de itens na coluna
# nunique: conta quantos valores diferentes cada coluna tem
faltantes = pd.DataFrame({
    'Faltantes': df_original.isna().sum(),
    '%': (df_original.isna().mean() * 100).round(2),
    'distintos': df_original.nunique(),
})

# filta somente as colunas que não possui faltantes
# sort e ascending: ordena pela coluna faltante do maior para o menor
faltantes[faltantes['Faltantes'] > 0].sort_values('Faltantes', ascending=False)

,Faltantes,%,distintos
DS_EMAIL,29322,100.00,0
NM_SOCIAL_CANDIDATO,29285,99.87,36
NM_FEDERACAO,24757,84.43,5
SG_FEDERACAO,24757,84.43,3
DS_COMPOSICAO_FEDERACAO,24757,84.43,3
DS_SIT_TOT_TURNO,1662,5.67,6
DT_NASCIMENTO,29,0.10,13495
DS_GENERO,29,0.10,2
DS_GRAU_INSTRUCAO,29,0.10,7
DS_ESTADO_CIVIL,29,0.10,5


Devido a baixíssima quantidades de faltantes, não é necessário a exclusão da coluna.

### 4.2 Outros tipos de faltantes
Analisando a tabela, verificamos que existe células preenchidas com texto que podem ser considerados 'faltantes'.

In [ ]:
nulos = pd.DataFrame({
    '#NULO': (df_original == '#NULO').sum(),
    '%': ((df_original == '#NULO').mean() * 100).round(2),
})

nulos[nulos['#NULO'] > 0].sort_values('#NULO', ascending=False)

In [ ]:
naodivulgavel = pd.DataFrame({
    'NÃO DIVULGÁVEL': (df_original == 'NÃO DIVULGÁVEL').sum(),
    '%': ((df_original == 'NÃO DIVULGÁVEL').mean() * 100).round(2),
})

naodivulgavel[naodivulgavel['NÃO DIVULGÁVEL'] > 0].sort_values('NÃO DIVULGÁVEL', ascending=False)

### 4.3. Substituindo por NaN
#NULO e NÃO DIVULGAVEL são strings, portanto é necessário transformar em NaN.

In [ ]:
df_original = df_original.replace('#NULO', np.nan)
df_original = df_original.replace('NÃO DIVULGÁVEL', np.nan)

### 4.4. Contagem de Faltantes após o tratamento

In [ ]:
faltantes_atualizado = pd.DataFrame({
    'Faltantes_atualizado': df_original.isna().sum(),
    '%': (df_original.isna().mean() * 100).round(2),
})

# filta somente as colunas que não possui faltantes
# sort e ascending: ordena pela coluna faltante do maior para o menor
faltantes_atualizado[faltantes_atualizado['Faltantes_atualizado'] > 0].sort_values('Faltantes_atualizado', ascending=False)

## 5. Tratamento de Duplicados
      

In [60]:
# verifica se existe linhas com todos os dados duplicados
print('Linhas totalmente duplicadas:', df_original.duplicated().sum())

Linhas totalmente duplicadas: 0


## 6. Tratamento da base auxiliar

Visando atender à exigência de uso do Merge e de três colunas numéricas, utilizou-se a tabela consulta_cand_2022_BRASIL, selecionando os campos NR_IDADE_DATA_POSSE e VR_DESPESA_MAX_CAMPANHA.

**Observação** = VR_DESPESA_MAX_CAMPANHA é o máximo que o candidato pode utilizar. Portanto, não quer dizer que foi utilizado tudo.

### 6.1. Lendo a base auxiliar

In [ ]:
# substitua o caminho abaixo pela pasta onde está salvo a base de dados .csv
# não esquecer que o nome do arquivo e a extensão deve estar inclusos

pasta_arquivo = "/Users/felipealbanez/Github/CAIXAVERSO_DFF/Projeto_Final_TSE_2022/consulta_cand_complementar_2022_BRASIL.csv"

# cria o dataframe lendo o arquivo .csv
auxiliar = pd.read_csv(pasta_arquivo, sep=';', decimal=',',encoding='latin-1')

# Seleciona a chave e apenas as duas colunas necessárias
# o SQ_CANDIDATO é número sequencial identificador único do candidato dentro da base do TSE
# funciona como um "id"
auxiliar = auxiliar[['SQ_CANDIDATO','NR_IDADE_DATA_POSSE', 'VR_DESPESA_MAX_CAMPANHA']].copy()

# transformando str em númemor
# parâmetro errors='coerce' diz: "tudo que não conseguir converter para número, transforme em NaN" em vez de travar o código com erro.
auxiliar['NR_IDADE_DATA_POSSE'] = pd.to_numeric(auxiliar['NR_IDADE_DATA_POSSE'], errors='coerce')
auxiliar['VR_DESPESA_MAX_CAMPANHA'] = pd.to_numeric(auxiliar['VR_DESPESA_MAX_CAMPANHA'], errors='coerce')

auxiliar.head()

,SQ_CANDIDATO,NR_IDADE_DATA_POSSE,VR_DESPESA_MAX_CAMPANHA
0,100001613736,49.0,1270629.01
1,160001597835,45.0,3176572.53
2,160001621917,41.0,3176572.53
3,110001643976,39.0,1270629.01
4,110001618165,46.0,1270629.01


In [100]:
# Identifica idades inválidas.
idades_invalidas = auxiliar[
    (auxiliar['NR_IDADE_DATA_POSSE'] < 18) |
    (auxiliar['NR_IDADE_DATA_POSSE'] > 100)
]
print('Idades inválidas:', len(idades_invalidas))

"""
# Substitui idades inválidas por NaN.
complementar.loc[
    (complementar['NR_IDADE_DATA_POSSE'] < 18) |
    (complementar['NR_IDADE_DATA_POSSE'] > 100),
    'NR_IDADE_DATA_POSSE'
] = np.nan
"""

# Identifica limites de despesas negativos.
despesas_invalidas = auxiliar[
    auxiliar['VR_DESPESA_MAX_CAMPANHA'] < 0
]
print('Despesas negativas:', len(despesas_invalidas))

"""
# Substitui despesas negativas por NaN.
complementar.loc[
    complementar['VR_DESPESA_MAX_CAMPANHA'] < 0,
    'VR_DESPESA_MAX_CAMPANHA'
] = np.nan
"""

Idades inválidas: 0
Despesas negativas: 859


"\n# Substitui despesas negativas por NaN.\ncomplementar.loc[\n    complementar['VR_DESPESA_MAX_CAMPANHA'] < 0,\n    'VR_DESPESA_MAX_CAMPANHA'\n] = np.nan\n"

## 7. Junçao das bases com Merge
**SQ_CANDIDATO** é um número sequencial identificador único do candidato dentro da base do TSE.

In [101]:
# merge com a chave SQ_CANDIDATO
df_merge = df_original.merge(auxiliar, on='SQ_CANDIDATO',how='left')

# dimensoes de antes e depois
print("Shape antes do merge:", df_original.shape)
print("Shape depois do merge:", df_merge.shape)

Shape antes do merge: (29322, 50)
Shape depois do merge: (29426, 52)


## 8. Criação de colunas
 Criação de colunas utilizando **np.where**, **pd.cut** e **pd.qcut**.

### 8.1 Coluna Eleito com **np.where**

In [102]:
# verificando os valores da coluna
print(df_merge['DS_SIT_TOT_TURNO'].value_counts(dropna=False))
print()

# criando a coluna ELEITO
df_merge['ELEITO'] = np.where(
    (df_merge['DS_SIT_TOT_TURNO'] == 'ELEITO') |
    (df_merge['DS_SIT_TOT_TURNO'] == 'ELEITO POR QP') |
    (df_merge['DS_SIT_TOT_TURNO'] == 'ELEITO POR MÉDIA'),'SIM','NÃO'
)

df_merge[['SQ_CANDIDATO','ELEITO']].head()

DS_SIT_TOT_TURNO
SUPLENTE            14623
NÃO ELEITO          11302
NaN                  1662
ELEITO POR QP        1182
ELEITO POR MÉDIA      390
ELEITO                163
2º TURNO              104
Name: count, dtype: int64



,SQ_CANDIDATO,ELEITO
0,250001597749,NÃO
1,10001595349,NÃO
2,230002529902,NÃO
3,140001596650,NÃO
4,100001599539,NÃO


### 8.2 Coluna Faixa Etária com **cut**

#### Candidato menor de 18 anos?

In [122]:
df_merge['FAIXA_ETARIA'] = pd.cut(df_merge['NR_IDADE_DATA_POSSE'], bins=[0, 18, 30, 45, 60, 80, 200],
                              labels=['<18', '18-30', '30-45', '45-60', '60-80', '80+'])
print(df_merge['FAIXA_ETARIA'].value_counts().sort_index())

# candidato com menos de 18 anos??

FAIXA_ETARIA
<18          1
18-30     1584
30-45    10246
45-60    13055
60-80     4437
80+         74
Name: count, dtype: int64


#### Alterando o Cut

In [123]:
# o cut por padrão é inclusivo no final
# ao incluir o "right=false", esse comportamento é alterado
df_merge['FAIXA_ETARIA'] = pd.cut(df_merge['NR_IDADE_DATA_POSSE'], bins=[0, 18, 30, 45, 60, 80, 200],
                              labels=['<18', '18-30', '30-45', '45-60', '60-80', '80+'], right = False) 
print(df_merge['FAIXA_ETARIA'].value_counts().sort_index())

FAIXA_ETARIA
<18          0
18-30     1319
30-45     9572
45-60    13350
60-80     5055
80+        101
Name: count, dtype: int64


### 8.3 Coluna com **qcut**

In [ ]:
df_merge['QUARTIL_DESPESA'] = pd.qcut(df_merge['VR_DESPESA_MAX_CAMPANHA'], q=4, labels=['Q1', 'Q2', 'Q3', 'Q4'])
print(df_merge['QUARTIL_DESPESA'].value_counts().sort_index())

#ERRO - TypeError: unsupported operand type(s) for -: 'str' and 'str'
# valores estão como str?
# transformar para número as novas colunas

# novo ERRO - ValueError: Bin edges must be unique: Index([-4.0, 1270629.01, 1270629.01, 3176572.53, 88944030.8], dtype='float64', name='VR_DESPESA_MAX_CAMPANHA').
# You can drop duplicate edges by setting the 'duplicates' kwarg




ValueError: Bin edges must be unique: Index([-4.0, 1270629.01, 1270629.01, 3176572.53, 88944030.8], dtype='float64', name='VR_DESPESA_MAX_CAMPANHA').
You can drop duplicate edges by setting the 'duplicates' kwarg